In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install optuna xgboost lightgbm "mlflow<3"

In [3]:
base_folder = "/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025"
%cd "{base_folder}"

/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025


In [4]:
import sqlite3
import pandas as pd

# base_folder should be defined in your environment (e.g., base_folder = ".")
conn = sqlite3.connect(f"{base_folder}/data/heart.db")

heart_data = pd.read_sql_query(
    """
    SELECT
        p.patient_id,
        p.age,
        p.sex,
        p.cp_id,
        p.trestbps,
        p.chol,
        p.fbs,
        p.ecg_id,
        p.thalach,
        p.exang,
        p.slope_id,
        p.ca,
        p.thal_id,
        d.target
    FROM patient AS p
    JOIN diagnosis AS d
        ON p.patient_id = d.patient_id
    ORDER BY p.patient_id
    """,
    conn,
)
conn.close()

# Display the first few rows to verify oldpeak is gone and target is joined
heart_data.head()

,patient_id,age,sex,cp_id,trestbps,chol,fbs,ecg_id,thalach,exang,slope_id,ca,thal_id,target
0,0,52,1,1,125,212,0,1,168,0,1,2,1,0
1,1,53,1,1,140,203,1,2,155,1,2,0,1,0
2,2,70,1,1,145,174,0,1,125,1,2,0,1,0
3,3,61,1,1,148,203,0,1,161,0,1,1,1,0
4,4,62,0,1,138,294,1,1,106,0,3,3,2,0


In [5]:
import time
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

import mlflow
from mlflow.models import infer_signature
import joblib

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna
from optuna.samplers import TPESampler
from sklearn.base import clone

# Shared components from your simplified heart_pipeline.py
from heart_pipeline import (
    build_preprocessing,
    make_estimator_for_name,
)

start_time = time.monotonic()
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =============================================================================
# STEP 1: Build Full ML Preprocessing Pipeline
# =============================================================================
# Contains ColumnDropper('patient_id') + selection of [ca, cp_id, thal_id]
preprocessing = build_preprocessing()
print("✓ STEP 1: Preprocessing pipeline created.")

# =============================================================================
# STEP 2: Split Data into Stratified Train and Test Sets
# =============================================================================
# Keep patient_id so the pipeline can drop it; helps maintain MLflow signature
X = heart_data.drop(["target"], axis=1)
y = heart_data["target"]

# Ensure integer columns are floats to avoid MLflow schema warnings
for col in ['ca', 'cp_id', 'thal_id']:
    if col in X.columns:
        X[col] = X[col].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"✓ STEP 2: Stratified split done. Train size: {len(X_train)}")

# =============================================================================
# STEP 3: Configure MLflow (Dagshub)
# =============================================================================
load_dotenv(
    dotenv_path="/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025/notebooks/.env",
    override=True
)

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("heart_disease_optuna_v3")

# =============================================================================
# STEP 4: Define Optuna Objective Functions (Classification - F1 Score)
# =============================================================================

def objective_logistic(trial, preprocessing, X_train, y_train):
    c_val = trial.suggest_float("logistic__C", 0.01, 10.0, log=True)
    pipeline = make_pipeline(clone(preprocessing), LogisticRegression(C=c_val, max_iter=1000))
    # We MAXIMIZE F1 score for classification
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring="f1", n_jobs=-1)
    return scores.mean()

def objective_hgb(trial, preprocessing, X_train, y_train):
    lr = trial.suggest_float("hgb__learning_rate", 0.01, 0.2)
    depth = trial.suggest_int("hgb__max_depth", 2, 5)
    pipeline = make_pipeline(clone(preprocessing), HistGradientBoostingClassifier(learning_rate=lr, max_depth=depth, random_state=42))
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring="f1", n_jobs=-1)
    return scores.mean()

def objective_xgb(trial, preprocessing, X_train, y_train):
    lr = trial.suggest_float("xgb__learning_rate", 0.01, 0.2)
    depth = trial.suggest_int("xgb__max_depth", 2, 5)
    pipeline = make_pipeline(clone(preprocessing), XGBClassifier(learning_rate=lr, max_depth=depth, random_state=42, n_jobs=-1))
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring="f1", n_jobs=-1)
    return scores.mean()

def objective_lgbm(trial, preprocessing, X_train, y_train):
    lr = trial.suggest_float("lgbm__learning_rate", 0.01, 0.2)
    leaves = trial.suggest_int("lgbm__num_leaves", 10, 31)
    pipeline = make_pipeline(clone(preprocessing), LGBMClassifier(learning_rate=lr, num_leaves=leaves, random_state=42, n_jobs=-1, verbose=-1))
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring="f1", n_jobs=-1)
    return scores.mean()

# =============================================================================
# STEP 5: Run Optuna Studies
# =============================================================================
model_names = ["logistic", "histgradientboosting", "xgboost", "lightgbm"]
objective_map = {
    "logistic": objective_logistic,
    "histgradientboosting": objective_hgb,
    "xgboost": objective_xgb,
    "lightgbm": objective_lgbm,
}

results = {}

for name in model_names:
    print(f"\nOptimizing {name}...")
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
    study.optimize(lambda trial: objective_map[name](trial, preprocessing, X_train, y_train), n_trials=10)

    # Reconstruct best pipeline
    best_params = study.best_params
    est = make_estimator_for_name(name)
    # Apply tuned params to the estimator
    if name == "logistic": est.set_params(C=best_params["logistic__C"])
    elif name == "histgradientboosting": est.set_params(learning_rate=best_params["hgb__learning_rate"], max_depth=best_params["hgb__max_depth"])
    elif name == "xgboost": est.set_params(learning_rate=best_params["xgb__learning_rate"], max_depth=best_params["xgb__max_depth"])
    elif name == "lightgbm": est.set_params(learning_rate=best_params["lgbm__learning_rate"], num_leaves=best_params["lgbm__num_leaves"])

    final_model = make_pipeline(clone(preprocessing), est)
    final_model.fit(X_train, y_train)

    # Eval
    y_pred = final_model.predict(X_test)
    test_f1 = f1_score(y_test, y_pred)

    results[name] = {"pipeline": final_model, "test_f1": test_f1, "cv_f1": study.best_value}

    with mlflow.start_run(run_name=f"{name}_optuna"):
        mlflow.log_params(best_params)
        mlflow.log_metric("f1_score", test_f1)
        mlflow.log_metric("cv_f1", study.best_value)
        signature = infer_signature(X_train, final_model.predict(X_train))
        mlflow.sklearn.log_model(final_model, "model", signature=signature)

# =============================================================================
# STEP 7: Global Best and Save
# =============================================================================
global_best_name = max(results, key=lambda k: results[k]["test_f1"])
global_best_pipeline = results[global_best_name]["pipeline"]

print(f"\nWINNER: {global_best_name} (F1: {results[global_best_name]['test_f1']:.4f})")

joblib.dump(global_best_pipeline, f"{base_folder}/models/global_best_heart_optuna.pkl")
print(f"Total time: {int((time.monotonic() - start_time) // 60)}m")

✓ STEP 1: Preprocessing pipeline created.
✓ STEP 2: Stratified split done. Train size: 820


2026/02/16 07:04:58 INFO mlflow.tracking.fluent: Experiment with name 'heart_disease_optuna_v3' does not exist. Creating a new experiment.



Optimizing logistic...


/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run logistic_optuna at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/1/runs/c55fa3216c204c2f9f42a8bc83f927b9
🧪 View experiment at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/1

Optimizing histgradientboosting...


...

🏃 View run histgradientboosting_optuna at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/1/runs/6cd2e49b4fe3499e81d3cb2c2ae90f25
🧪 View experiment at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/1

Optimizing xgboost...


/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run xgboost_optuna at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/1/runs/ae1e5980fe56495e9ea7b8da44536770
🧪 View experiment at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/1

Optimizing lightgbm...
[LightGBM] [Info] Number of positive: 421, number of negative: 399
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000081 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 20
[LightGBM] [Info] Number of data points in the train set: 820, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.513415 -> initscore=0.053671
[LightGBM] [Info] Start training from score 0.053671


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing

🏃 View run lightgbm_optuna at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/1/runs/e802bd28492b4810b285333ecf666857
🧪 View experiment at: https://dagshub.com/atom2binghamton/heart_diagnosis_fall2025.mlflow/#/experiments/1

WINNER: xgboost (F1: 0.8704)
Total time: 0m
